In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset, Dataset, WeightedRandomSampler
import os
import numpy as np
import pandas as pd
import nibabel as nib
import sys
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from util import filter_data
from util import mask_crop as mask_crop_fn
from validate import val_model_stable as val_model

# ============================================================
# DEBUG & CACHE CONFIG
# ============================================================
LIMIT_SUBS = None 
CACHE_PATH = 'qsm_preprocessed_cache.pt'
LOAD_FROM_CACHE = True  # Set to True only if cache exists

# ============================================================
# 1. RAM-BASED DATASET CLASS (LOADS ALL 111 SUBJECTS)
# ============================================================
# ============================================================
# 1. RAM-BASED DATASET CLASS (LOADS ALL 111 SUBJECTS)
# ============================================================
class QSM_RAM_Dataset(Dataset):
    def __init__(self, nii_dir, seg_dir, mask_crop_fn, clinical_dict, label_map, limit=None, cache_path=None, load_cache=False):
        self.samples = []
        self.volumes = {} 
        self.clinical_dict = clinical_dict
        self.label_map = label_map
        
        # --- NEW ATTRIBUTES FOR AUGMENTATION ---
        self.transform = None   # Will hold the transforms.Compose object
        self.train_mode = False # Toggle to True during training phase
        
        # 1. LOAD FROM CACHE
        if load_cache and cache_path and os.path.exists(cache_path):
            print(f">>> Loading preprocessed data from cache: {cache_path}...")
            cached_data = torch.load(cache_path)
            self.volumes = cached_data['volumes']
            self.samples = cached_data['samples']
            print(f">>> Cache Loaded: {len(self.volumes)} subjects.")
            return 

        # 2. PROCESS NIFTI (If no cache)
        all_potential = [f for f in os.listdir(nii_dir) if f.startswith('qsm_') and f.endswith('.nii.gz')]
        print(f">>> Cache not found. Processing {len(all_potential)} volumes...")
        
        loaded_count = 0
        for f in tqdm(all_potential, desc="Caching Volumes"):
            if limit and loaded_count >= limit: break
            try:
                sub_id = int(f.split('_')[1])
                case_id = f"{sub_id:02d}"
                mask_path = os.path.join(seg_dir, f'seg_{case_id}.nii.gz')
                if not os.path.exists(mask_path): continue
                
                raw_data = nib.load(os.path.join(nii_dir, f)).get_fdata()
                mask_data = nib.load(mask_path).get_fdata()
                mask_data[mask_data <= 2] = 0
                binary_mask = (mask_data > 0).astype(np.uint8)
                
                img = mask_crop_fn(raw_data, mask_data, (72, 64, 64)) / 1000.0
                m_patch = mask_crop_fn(binary_mask, mask_data, (72, 64, 64))
                
                if img.shape != (72, 64, 64): continue
                brain_indices = m_patch > 0
                if np.any(brain_indices):
                    img = (img - np.mean(img[brain_indices])) / (np.std(img[brain_indices]) + 1e-8)
                
                processed_vol = np.transpose(np.clip(img, -5.0, 5.0), (1, 2, 0)).astype(np.float32)
                self.volumes[sub_id] = processed_vol
                
                actual_label = self.label_map.get(sub_id, -1)
                for slice_idx in range(72):
                    self.samples.append({'sub_id': sub_id, 'slice_idx': slice_idx, 'label': actual_label})
                loaded_count += 1
            except Exception: continue
        
        # 3. SAVE TO CACHE
        if cache_path:
            print(f">>> Saving preprocessed data to cache: {cache_path}...")
            torch.save({'volumes': self.volumes, 'samples': self.samples}, cache_path)

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, index):
        meta = self.samples[index]
        vol = self.volumes[meta['sub_id']]
        
        # Extract the specific slice
        img = vol[:, :, meta['slice_idx']]
        img_tensor = torch.from_numpy(img).unsqueeze(0) # Shape: [1, 64, 64]
        
        # --- APPLY ON-THE-FLY AUGMENTATION ---
        if self.train_mode and self.transform:
            img_tensor = self.transform(img_tensor)
        
        # Normalize the slice from [-5, 5] back to [0, 1] range roughly for ResNet
        img_tensor = img_tensor / 5.0 
        
        clin_data = self.clinical_dict.get(str(meta['sub_id']))
        if clin_data is None:
            # Fallback for subjects missing clinical data (unlabeled pool)
            clin_vec = torch.zeros(getattr(self, 'clin_dim', 11), dtype=torch.float32)
        else:
            clin_vec = torch.tensor(clin_data, dtype=torch.float32)
            
        return img_tensor, clin_vec, int(meta['label'])
# ============================================================
# 2. MODELS
# ============================================================
class QSMDecoder(nn.Module):
    def __init__(self, feat_dim=512):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(feat_dim, 256, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.decoder(x)

class ResNetWrapper(nn.Module):
    def __init__(self, model, clinical_dim):
        super().__init__()
        self.base_model = model
        self.feat_dim = model.fc.in_features
        self.base_model.fc = nn.Identity()
        self.fusion = nn.Sequential(
            nn.Linear(self.feat_dim + clinical_dim, 256), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(256, 2)
        )
    def forward(self, x, clinical_vec):
        feats = self.base_model(x)
        return self.fusion(torch.cat([feats, clinical_vec], dim=1)), feats

# ============================================================
# 3. RUNTIME & ZERO-LEAKAGE CV
# ============================================================
device = 'cuda:3'
nii_path = '/media/mts_dbs/dbs/all/nii/qsm_115/im'
seg_path = '/media/mts_dbs/dbs/all/nii/seg_ps/'
file_dir = '/data/Ali/RadDBS-QSM/data/docs/dbs_03292024.csv'

# ============================================================
# 3. RUNTIME & ZERO-LEAKAGE CV (UPDATED)
# ============================================================

# 1. Define only BASELINE features (No 'OFF meds ON stim 6mo')
cv_features = {
    'Age', 'Sex', 'Ethnicity', 'Race', 'Disease Duration (year)', 
    'Physician', ' pre op levadopa equivalent dose (mg)', 
    ' Location', ' Target', ' Test medication status',
    ' ON (pre-dbs updrs)' # This is baseline, so it's okay.
}

# 2. Keep the calculation columns separate just for labeling
all_needed_cols = cv_features | {'CORNELL ID', ' OFF meds ON stim 6mo'}

motor_df = filter_data(file_dir, all_needed_cols, True)
motor_df[' ON (pre-dbs updrs)'] = pd.to_numeric(motor_df[' ON (pre-dbs updrs)'], errors='coerce')
motor_df[' OFF meds ON stim 6mo'] = pd.to_numeric(motor_df[' OFF meds ON stim 6mo'], errors='coerce')
motor_df = motor_df.dropna(subset=[' ON (pre-dbs updrs)', ' OFF meds ON stim 6mo'])

# Ratio calculation (Labeling only)
improvement_ratios = (motor_df[' ON (pre-dbs updrs)'] - motor_df[' OFF meds ON stim 6mo']) / motor_df[' ON (pre-dbs updrs)']
label_map = {int(row['CORNELL ID']): (1 if ratio >= 0.30 else 0) for (_, row), ratio in zip(motor_df.iterrows(), improvement_ratios)}

# 3. Clinical Dictionary - ONLY include the features, EXCLUDE the post-op label info
clinical_dict = {
    str(int(row['CORNELL ID'])): row[list(cv_features)].values.astype(np.float32) 
    for _, row in motor_df.iterrows()
}

# Detect Clinical Dimension (should be 11 or 12 now, not 13)
sample_key = next(iter(clinical_dict))
actual_clin_dim = clinical_dict[sample_key].shape[0]
print(f">>> Cleaned Clinical Features: {list(cv_features)}")
print(f">>> New Feature Dimension: {actual_clin_dim}")
# Pre-loading
# To THIS:
full_dataset = QSM_RAM_Dataset(
    nii_path, 
    seg_path, 
    mask_crop_fn, 
    clinical_dict, 
    label_map, 
    limit=LIMIT_SUBS,
    cache_path=CACHE_PATH,     # Pass the string 'qsm_preprocessed_cache.pt'
    load_cache=LOAD_FROM_CACHE # Pass True
)
sample_key = next(iter(clinical_dict))
actual_clin_dim = clinical_dict[sample_key].shape[0]
full_dataset.clin_dim = actual_clin_dim

# Partitioning IDs
all_cached_ids = set(full_dataset.volumes.keys())
labeled_ids = set(label_map.keys())
unlabeled_ids = np.array(list(all_cached_ids - labeled_ids))
labeled_subs = np.array(list(all_cached_ids & labeled_ids))
sub_labels = np.array([label_map[sid] for sid in labeled_subs])

# Define the augmentation pipeline
# We use RandomRotation for head tilt and RandomAffine for slight positioning shifts
qsm_aug = transforms.Compose([
    transforms.RandomRotation(degrees=15),  # Rotate up to 15 degrees in either direction
    transforms.RandomAffine(
        degrees=0, 
        translate=(0.05, 0.05), # Horizontal/Vertical shifts (5% of image size)
        scale=(0.95, 1.05)      # Slight zoom in/out (5%)
    )
])
all_split_best_metrics = []
if len(labeled_subs) >= 5:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    class_counts = np.bincount(sub_labels)
    weights = torch.tensor([len(sub_labels)/(2*class_counts[0]), len(sub_labels)/(2*class_counts[1])], dtype=torch.float32).to(device)
    
    for split, (t_p_idx, v_p_idx) in enumerate(skf.split(labeled_subs, sub_labels)):
        train_subs, val_subs = labeled_subs[t_p_idx], labeled_subs[v_p_idx]
        
        # LEAKAGE PROTECTION: Pretraining pool only includes Unlabeled + THIS split's Training subjects
        pt_subs = np.concatenate([unlabeled_ids, train_subs])
        
        pt_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in pt_subs]
        t_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in train_subs]
        v_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in val_subs]

        # --- RE-DEFINING ALL LOADERS ---
        
        # 1. Pretraining Loader (Unlabeled + current Train subjects)
        pt_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in pt_subs]
        pt_loader = DataLoader(Subset(full_dataset, pt_idx), batch_size=48, shuffle=True)

        # 2. Training Loader (BALANCED using WeightedRandomSampler)
        train_subset = Subset(full_dataset, t_idx)
        train_labels = [full_dataset.samples[i]['label'] for i in t_idx]
        class_counts = np.bincount(train_labels)
        class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
        sample_weights = [class_weights[l] for l in train_labels]
        sampler = WeightedRandomSampler(sample_weights, 2*len(sample_weights))
        t_loader = DataLoader(train_subset, batch_size=48, sampler=sampler)

        # 3. Validation Loader (Normal)
        v_idx = [i for i, s in enumerate(full_dataset.samples) if s['sub_id'] in val_subs]
        v_loader = DataLoader(Subset(full_dataset, v_idx), batch_size=48, shuffle=False)

        print(f"\n>>> Split {split} | PT Subjects: {len(pt_subs)} | Val Subjects: {len(val_subs)}")

        # --- A. FOLD-SPECIFIC PRETRAINING ---
        base_resnet = models.resnet18(pretrained=True)
        base_resnet.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        encoder = nn.Sequential(*list(base_resnet.children())[:-2]).to(device)
        decoder = QSMDecoder().to(device)
        
        optimizer_pt = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-4)
        criterion_pt = nn.MSELoss()
        full_dataset.train_mode = True
        full_dataset.transform = qsm_aug
        for pt_epoch in range(5): # Fast pretraining per fold
            encoder.train(); decoder.train()
            for imgs, _, _ in pt_loader:
                imgs = imgs.to(device)
                optimizer_pt.zero_grad()
                recon = decoder(encoder(imgs))
                loss = criterion_pt(recon, imgs)
                loss.backward(); optimizer_pt.step()

        # --- B. FINE-TUNING ---
        model = ResNetWrapper(base_resnet, clinical_dim=actual_clin_dim).to(device)
        for param in model.base_model.parameters(): param.requires_grad = False # Freeze backbone
        
        optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-3)
        loss_fn = nn.CrossEntropyLoss().to(device)

        best_f1, patience = 0, 0
        best_metrics_this_split = None  # Reset for each split

        for epoch in range(30):
            full_dataset.train_mode = True
            full_dataset.transform = qsm_aug # Define this outside the loop
            model.train()
            for imgs, clin, lbls in t_loader:
                imgs, clin, lbls = imgs.to(device), clin.to(device), lbls.to(device)
                optimizer.zero_grad()
                logits, _ = model(imgs, clin)
                loss = loss_fn(logits, lbls)
                loss.backward(); optimizer.step()
            
            full_dataset.train_mode = False
            full_dataset.transform = None # Define this outside the loop
            model.eval()
            m = val_model(v_loader, device, model, loss_fn, v_loader.dataset, threshold=0.5)
            v_loss, v_acc, v_prec, v_sens, v_spec, v_auc = m

            # Calculate F1-Score
            current_f1 = 2 * (v_prec * v_sens) / (v_prec + v_sens) if (v_prec + v_sens) > 0 else 0

            # SAVE BEST BASED ON F1
            if current_f1 > best_f1:
                best_f1 = current_f1
                best_metrics_this_split = m  # Capture full array
                patience = 0
                torch.save(model.state_dict(), f"best_f1_model_split_{split}.pth")
            else:
                patience += 1

            print(f"Split {split} Ep {epoch} | F1: {current_f1:.4f} | AUC: {v_auc:.4f} | Acc: {v_acc:.4f}")
            print(f"Prec: {v_prec:.4f} | Sens: {v_sens:.4f} | Spec: {v_spec:.4f}")
            print("-" * 30)
            
            if patience >= 10: break
        
        # IMPORTANT: Append only the BEST version from this split once the epoch loop finishes
        if best_metrics_this_split is not None:
            all_split_best_metrics.append(best_metrics_this_split)

# ============================================================
# FINAL SUMMARY ACROSS ALL SPLITS
# ============================================================
final_metrics = np.array(all_split_best_metrics)
avg_metrics = np.mean(final_metrics, axis=0)
std_metrics = np.std(final_metrics, axis=0)

print("\n" + "="*45)
print("FINAL CV SUMMARY (BEST F1 PER SPLIT)")
print("="*45)
names = ["Loss", "Accuracy", "Precision", "Sensitivity", "Specificity", "AUC"]
for i, name in enumerate(names):
    print(f"{name:<15} : {avg_metrics[i]:.4f} ± {std_metrics[i]:.4f}")
print("="*45)

/data/Ali/anaconda3/envs/bigan/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Keeping CORNELL ID
Keeping Age
Keeping Sex
Keeping Ethnicity
Keeping Race
Keeping Disease Duration (year)
Keeping Physician
Keeping  ON (pre-dbs updrs)
Keeping  pre op levadopa equivalent dose (mg)
Keeping  Location
Keeping  Target
Keeping  Test medication status
Keeping  OFF meds ON stim 6mo
>>> Cleaned Clinical Features: ['Race', ' Location', 'Ethnicity', ' Target', ' ON (pre-dbs updrs)', ' pre op levadopa equivalent dose (mg)', 'Age', ' Test medication status', 'Disease Duration (year)', 'Physician', 'Sex']
>>> New Feature Dimension: 11
>>> Loading preprocessed data from cache: qsm_preprocessed_cache.pt...
>>> Cache Loaded: 108 subjects.

>>> Split 0 | PT Subjects: 94 | Val Subjects: 14
Split 0 Ep 0 | F1: 0.7785 | AUC: 0.8920 | Acc: 0.7103
Prec: 0.6914 | Sens: 0.8906 | Spec: 0.4699
------------------------------
Split 0 Ep 1 | F1: 0.7814 | AUC: 0.8869 | Acc: 0.7202
Prec: 0.7059 | Sens: 0.8750 | Spec: 0.5139
------------------------------
Split 0 Ep 2 | F1: 0.7861 | AUC: 0.8615 | Acc